# Bearing Capacity Check (Terzaghi + EC7 DA1)

This notebook performs shallow foundation bearing checks using the Terzaghi general equation with EC7 Design Approach 1 combinations:
- `DA1-1 = A1 + M1 + R1`
- `DA1-2 = A2 + M2 + R1`

Assumptions in this template:
- Vertical centric load only (`i` factors = 1.0)
- Drained parameters (`c'`, `phi'`)
- Effective stress method
- Rectangular footing with Terzaghi-style shape/depth factors


In [ ]:
from calc_setup import *
import pandas as pd
from math import atan, degrees


## Inputs
Use SI base units: `kN`, `m`, `kN/m^3`, `kPa`.


In [ ]:
# Geometry
B = 2.0         # footing width (m)
L = 3.0         # footing length (m)
Df = 1.2        # foundation embedment depth (m)

# Characteristic actions
Gk = 900.0      # permanent vertical load (kN, unfavorable)
Qk = 300.0      # variable vertical load (kN, unfavorable)

# Ground / soil (characteristic)
gamma_eff = 18.0   # effective unit weight (kN/m^3)
c_k = 5.0          # effective cohesion c' (kPa = kN/m^2)
phi_k_deg = 32.0   # effective friction angle phi' (deg)

# Resistance factor set (R1)
gamma_Rv = 1.0

# EC7 factors (edit if your National Annex differs)
A1 = {"gamma_G": 1.35, "gamma_Q": 1.50}
A2 = {"gamma_G": 1.00, "gamma_Q": 1.30}
M1 = {"gamma_c": 1.00, "gamma_phi": 1.00}
M2 = {"gamma_c": 1.25, "gamma_phi": 1.25}


In [ ]:
def bearing_factors_terzaghi(phi_deg: float):
    """Terzaghi bearing capacity factors (approximate N_gamma form)."""
    phi = radians(phi_deg)
    if abs(phi) < 1e-9:
        Nq = 1.0
        Nc = 5.14
        Ngamma = 0.0
    else:
        Nq = exp(pi * tan(phi)) * tan(radians(45.0) + phi / 2.0) ** 2
        Nc = (Nq - 1.0) / tan(phi)
        Ngamma = 1.5 * (Nq - 1.0) * tan(phi)
    return Nc, Nq, Ngamma

def shape_depth_factors(B: float, L: float, Df: float):
    beta = B / L
    sc = 1.0 + 0.2 * beta
    sq = 1.0 + 0.1 * beta
    sgamma = max(0.6, 1.0 - 0.4 * beta)

    eta = Df / B
    dc = 1.0 + 0.2 * eta
    dq = 1.0 + 0.1 * eta
    dgamma = 1.0

    ic = iq = igamma = 1.0
    return sc, sq, sgamma, dc, dq, dgamma, ic, iq, igamma

def q_ult_terzaghi(c_d, phi_d_deg, gamma_eff, B, Df, L):
    Nc, Nq, Ngamma = bearing_factors_terzaghi(phi_d_deg)
    sc, sq, sgamma, dc, dq, dgamma, ic, iq, igamma = shape_depth_factors(B, L, Df)
    q_overburden = gamma_eff * Df

    term_c = c_d * Nc * sc * dc * ic
    term_q = q_overburden * Nq * sq * dq * iq
    term_gamma = 0.5 * gamma_eff * B * Ngamma * sgamma * dgamma * igamma
    q_ult = term_c + term_q + term_gamma

    return {
        "Nc": Nc,
        "Nq": Nq,
        "Ngamma": Ngamma,
        "q_overburden": q_overburden,
        "term_c": term_c,
        "term_q": term_q,
        "term_gamma": term_gamma,
        "q_ult": q_ult,
    }

def design_check(label, action_factors, material_factors):
    V_Ed = action_factors["gamma_G"] * Gk + action_factors["gamma_Q"] * Qk
    A = B * L
    q_Ed = V_Ed / A

    c_d = c_k / material_factors["gamma_c"]
    tan_phi_d = tan(radians(phi_k_deg)) / material_factors["gamma_phi"]
    phi_d_deg = degrees(atan(tan_phi_d))

    bc = q_ult_terzaghi(c_d, phi_d_deg, gamma_eff, B, Df, L)
    q_Rd = bc["q_ult"] / gamma_Rv

    util = q_Ed / q_Rd
    ok = util <= 1.0

    out = {
        "Case": label,
        "phi_d_deg": phi_d_deg,
        "c_d_kPa": c_d,
        "V_Ed_kN": V_Ed,
        "q_Ed_kPa": q_Ed,
        "q_ult_kPa": bc["q_ult"],
        "q_Rd_kPa": q_Rd,
        "utilization": util,
        "PASS": ok,
    }
    out.update({f"bc_{k}": v for k, v in bc.items()})
    return out


## DA1-1 (`A1 + M1 + R1`)


In [ ]:
%%render
da11 = design_check("DA1-1 (A1+M1+R1)", A1, M1)
V_{Ed} = A1["gamma_G"] * Gk + A1["gamma_Q"] * Qk
A = B * L
q_{Ed} = V_{Ed} / A

c_d = c_k / M1["gamma_c"]
tanphi_d = tan(radians(phi_k_deg)) / M1["gamma_phi"]
phi_d = degrees(atan(tanphi_d))

q_{ult} = da11["q_ult_kPa"]
q_{Rd} = q_{ult} / gamma_Rv
util = q_{Ed} / q_{Rd}


## DA1-2 (`A2 + M2 + R1`)


In [ ]:
%%render
da12 = design_check("DA1-2 (A2+M2+R1)", A2, M2)
V_{Ed} = A2["gamma_G"] * Gk + A2["gamma_Q"] * Qk
A = B * L
q_{Ed} = V_{Ed} / A

c_d = c_k / M2["gamma_c"]
tanphi_d = tan(radians(phi_k_deg)) / M2["gamma_phi"]
phi_d = degrees(atan(tanphi_d))

q_{ult} = da12["q_ult_kPa"]
q_{Rd} = q_{ult} / gamma_Rv
util = q_{Ed} / q_{Rd}


## Summary Table


In [ ]:
summary = pd.DataFrame([da11, da12])[
    ["Case", "phi_d_deg", "c_d_kPa", "V_Ed_kN", "q_Ed_kPa", "q_Rd_kPa", "utilization", "PASS"]
]
summary


## Notes
- If your National Annex uses different partial factors, edit `A1`, `A2`, `M1`, and `M2` in the input cell.
- This notebook checks bearing resistance only. Settlement and sliding/overturning must be checked separately.
- For eccentric or inclined loading, add inclination/eccentricity factors before finalizing design resistance.
